# Évaluer une classification sans se limiter à l’accuracy

**Objectifs.** Construire une baseline, lire une confusion matrix, relier précision/rappel au seuil, et comparer des modèles sans toucher au test pendant le tuning. Durée : 75–100 min.

## 1. Une classification binaire et sa baseline

Le dataset `breast_cancer` est intégré à scikit-learn. Dans cette leçon on interprète la classe `1` telle que fournie ; ce choix est pédagogique et n’a aucune portée de diagnostic. On commence par un split stratifié.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score, precision_recall_curve, average_precision_score)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=21)
baseline = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=2_000))
model.fit(x_train, y_train)
print('Baseline accuracy :', round(accuracy_score(y_test, baseline.predict(x_test)), 3))
print('Modèle accuracy :', round(accuracy_score(y_test, model.predict(x_test)), 3))

## 2. Confusion matrix et métriques

`TN, FP, FN, TP` signifient vrais négatifs, faux positifs, faux négatifs et vrais positifs. La précision parle des alertes produites, le rappel des positifs trouvés. Quelle erreur coûte le plus dépend de l’usage.

In [ ]:
test_prediction = model.predict(x_test)
cm = confusion_matrix(y_test, test_prediction)
ConfusionMatrixDisplay(cm, display_labels=data.target_names).plot(cmap='Blues', values_format='d')
plt.title('Comptes sur le test'); plt.show()
print('precision :', round(precision_score(y_test, test_prediction), 3))
print('recall    :', round(recall_score(y_test, test_prediction), 3))
print('F1        :', round(f1_score(y_test, test_prediction), 3))

## 3. Le seuil modifie les erreurs

La régression logistique retourne des scores probabilistes. Le seuil 0.5 est conventionnel, pas magique. Le modifier change les alertes, donc FP et FN ; le test ne doit pas être utilisé pour choisir ce seuil. Cette visualisation est descriptive et ne propose pas de seuil clinique.

In [ ]:
probability = model.predict_proba(x_test)[:, 1]
for threshold in [0.3, 0.5, 0.7]:
    prediction_at_threshold = (probability >= threshold).astype(int)
    print(f'seuil={threshold:.1f} precision={precision_score(y_test, prediction_at_threshold):.3f} recall={recall_score(y_test, prediction_at_threshold):.3f}')
precision_values, recall_values, thresholds = precision_recall_curve(y_test, probability)
plt.plot(recall_values, precision_values)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Courbe precision–recall (split de démonstration)')
plt.grid(alpha=0.2); plt.show()
print('Average precision :', round(average_precision_score(y_test, probability), 3))

## 4. Validation croisée pour comparer

La cross-validation utilise plusieurs partitions train/validation, sans toucher au test final. La pipeline entière est ajustée dans chaque fold, donc l’imputation et le scaling n’aperçoivent pas la validation de ce fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
fold_scores = cross_val_score(model, x_train, y_train, cv=cv, scoring='average_precision')
print('AP par fold :', fold_scores.round(3))
print('Moyenne ± écart-type :', round(fold_scores.mean(), 3), '±', round(fold_scores.std(), 3))

## 5. Examiner les erreurs et les limites

Une métrique résume un groupe d’exemples. Avant d’agir, examiner cas mal classés, calibrage, provenance et représentativité des données. Les mesures du même dataset peuvent être dépendantes ; une stratification simple ne corrige pas toutes les formes de dépendance. ROC-AUC ou PR-AUC évaluent classement, pas le coût réel ni calibration.

In [ ]:
error_table = x_test.copy()
error_table['actual'] = y_test
error_table['predicted'] = test_prediction
error_table['probability_class_1'] = probability
misclassified = error_table[error_table['actual'] != error_table['predicted']]
print('Nombre d’erreurs :', len(misclassified))
misclassified[['actual', 'predicted', 'probability_class_1']].head()

## À retenir et exercices

Prévoir une règle de décision, mesurer les erreurs associées, puis écrire la métrique principale et les coûts qu’elle ne capture pas. Garder le test final hors du réglage. Exercices : [`05_evaluation.ipynb`](../../exercises/intermediate/05_evaluation.ipynb).